# Homework 1 — Submission: blend of mlp_4out_emb and LightGBM

The blend chosen in [ensemble.ipynb](ensemble.ipynb): **0.65 · mlp_4out_emb + 0.35 · lgbm_building**, averaged in
`log1p` space. On the 4 seasonal folds it scored 1.178 against 1.187 for the MLP alone.

- **Training rows:** 2M sampled from all of 2016, anomalies dropped. The fold experiments trained on 2M, and 4M didn't
  help ([ensemble.ipynb, More training rows](ensemble.ipynb)).
- **mlp_4out_emb:** 2 hidden layers of 64, 16-number building embedding, Adam, 20 epochs
  ([epochs_test.ipynb](epochs_test.ipynb)).
- **lgbm_building:** LightGBM on the same 33 features plus `building_id` as a categorical (500 trees, learning rate 0.1,
  63 leaves).

Output: `data/processed/submission_blend.csv`, with site 0 electricity converted back to kBTU, the unit Kaggle scores.

In [1]:
import sys

import mlflow
import numpy as np
import pandas as pd
import torch

sys.path.insert(0, "..")  # the ashrae package lives in the homework folder
from ashrae.config import DATA, KBTU_TO_KWH, PROCESSED
from ashrae.data import read_processed
from ashrae.features import build_features
from ashrae.modeling import MeterMLP, make_loaders
from ashrae.tracking import Experiment

# Imported after torch: loading LightGBM first crashes the kernel on macOS (both ship an OpenMP runtime).
import lightgbm as lgb

N_TRAIN = 2_000_000
EPOCHS = 20
W_MLP = 0.65  # MLP weight in the blend, from ensemble.ipynb
LGBM_PARAMS = {"n_estimators": 500, "learning_rate": 0.1, "num_leaves": 63}
experiment = Experiment("ashrae-submission")

2026/09/27 21:53:14 INFO mlflow.tracking.fluent: Experiment with name 'ashrae-submission' does not exist. Creating a new experiment.


MLflow: sqlite:////Users/lupaskodmitro/FuckIT/Machine Learning/Deep Learning Course/Homeworks/Homework 1/mlflow.db, experiment 'ashrae-submission', session 2026-09-27 21:53:14


## Training rows

Sampled from all of 2016 without anomalies ([EDA §9](../eda/EDA.ipynb#broken)). The `year_built` median and the
feature scaler come from these rows and are reused for the test set.

In [2]:
df = read_processed()
n_buildings = int(df.building_id.max()) + 1
BUILDING = pd.CategoricalDtype(range(n_buildings))
train_df = df[~df.anomaly].sample(N_TRAIN, random_state=0)
del df
year_median = train_df.year_built.median()
X_train = build_features(train_df, year_median)
y_train = np.log1p(train_df.meter_reading.to_numpy(dtype="float32"))
print(f"{len(train_df):,} training rows, {X_train.shape[1]} features, "
      f"{train_df.timestamp.dt.month.value_counts().min():,}–{train_df.timestamp.dt.month.value_counts().max():,} per month")


def with_building(X, rows):
    return X.assign(building_id=rows.building_id.astype(int).astype(BUILDING).to_numpy())

2,000,000 training rows, 33 features, 143,133–175,101 per month


## Models

In [3]:
train_loader, _, scaler = make_loaders(train_df, None, X_train, None)
mlp, run_id = experiment.fit(lambda: MeterMLP(X_train.shape[1], n_buildings), train_loader,
                        run_name=f"mlp_4out_emb | {N_TRAIN:,} rows", epochs=EPOCHS, tags={"model": "mlp_4out_emb"})
mlp = mlp.eval()
print(f"mlp_4out_emb: training RMSLE {experiment.history(run_id, 'train_rmsle')[-1]:.3f}")

/Users/lupaskodmitro/FuckIT/Machine Learning/Deep Learning Course/.venv/lib/python3.14/site-packages/lightning/pytorch/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.


mlp_4out_emb: training RMSLE 0.819


In [4]:
# n_jobs=1: multi-threaded LightGBM segfaults once torch has loaded its OpenMP runtime
lgbm = lgb.LGBMRegressor(**LGBM_PARAMS, random_state=0, verbose=-1, n_jobs=1).fit(with_building(X_train, train_df), y_train)
with mlflow.start_run(run_name=f"lgbm_building | {N_TRAIN:,} rows",
                      tags={"session": experiment.session, "model": "lgbm_building"}):
    mlflow.log_params(LGBM_PARAMS)
del train_loader

## Test predictions

Both models predict the 41.7M test rows in chunks of 1M. The `log1p` predictions are clipped at 0, blended, and turned
back into readings with `expm1`.

In [5]:
test_df = read_processed("test_merged.parquet")


@torch.no_grad()
def predict_chunk(chunk):
    """log1p predictions of both models for the rows of `chunk`."""
    X = build_features(chunk, year_median).reindex(columns=X_train.columns, fill_value=0)
    batch = {
        "x": torch.tensor(scaler.transform(X), dtype=torch.float32, device=mlp.device),
        "meter": torch.tensor(chunk.meter.to_numpy(), dtype=torch.long, device=mlp.device),
        "building": torch.tensor(chunk.building_id.to_numpy(), dtype=torch.long, device=mlp.device),
    }
    return mlp(batch).clamp(min=0).cpu().numpy(), np.clip(lgbm.predict(with_building(X, chunk)), 0, None)


chunks = [predict_chunk(test_df.iloc[i : i + 1_000_000]) for i in range(0, len(test_df), 1_000_000)]
mlp_log = np.concatenate([c[0] for c in chunks])
lgbm_log = np.concatenate([c[1] for c in chunks])
del chunks
print(f"correlation of the two models' test predictions (log1p): {np.corrcoef(mlp_log, lgbm_log)[0, 1]:.3f}")
print(f"mean log1p prediction: mlp {mlp_log.mean():.3f}, lgbm {lgbm_log.mean():.3f}")

correlation of the two models' test predictions (log1p): 0.976
mean log1p prediction: mlp 4.244, lgbm 4.234


In [6]:
reading = np.expm1(W_MLP * mlp_log + (1 - W_MLP) * lgbm_log)
reading[((test_df.site_id == 0) & (test_df.meter == 0)).to_numpy()] /= KBTU_TO_KWH  # back to kBTU

submission = pd.DataFrame({"row_id": test_df.row_id.to_numpy(), "meter_reading": reading}).sort_values("row_id")
sample_ids = pd.read_csv(DATA / "sample_submission.csv", usecols=["row_id"]).row_id
assert np.array_equal(submission.row_id, sample_ids) and np.isfinite(submission.meter_reading).all()
assert (submission.meter_reading >= 0).all()
submission.to_csv(PROCESSED / "submission_blend.csv", index=False, float_format="%.4f")
print(f"saved {PROCESSED / 'submission_blend.csv'}: {len(submission):,} rows")
submission.meter_reading.describe().round(2).to_frame()

saved /Users/lupaskodmitro/FuckIT/Machine Learning/Deep Learning Course/Homeworks/Homework 1/data/processed/submission_blend.csv: 41,697,600 rows


,meter_reading
count,41697600.00
mean,400.83
std,2392.63
min,0.00
25%,21.99
50%,79.36
75%,245.16
max,496449.30


## Next

No leaderboard score yet. Uploading `submission_blend.csv` (and `submission_mlp.csv` for comparison) to Kaggle is the
only check that the fold validation tracks the test years.